In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.preprocessing import OrdinalEncoder
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import accuracy_score, recall_score, roc_auc_score, f1_score
from snowflake.ml.registry import Registry
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
import pandas as pd
import datetime

DB = "HACKATHON"
SCHEMA = "PUBLIC"
GOLD_TABLE = f"{DB}.{SCHEMA}.CHURN_GOLD"
FS_DATASET_NAME = "CHURN_TRAIN_SET"
MODEL_NAME = "CFPB_CHURN_PREDICTOR"

session = get_active_session()
session.sql(f"USE DATABASE {DB}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"CREATE STAGE IF NOT EXISTS {DB}.{SCHEMA}.SNOWPARK_TEMP_STAGE").collect()

In [ ]:
# 1. Dynamic Extraction from Marketplace
raw_df = session.table("SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FINANCIAL_CFPB_COMPLAINT")

# 2. Filtering & Labeling
processed_df = raw_df.filter(
    (F.lower(F.col("PRODUCT")).like("%credit card%")) |
    (F.lower(F.col("PRODUCT")).like("%bank account%"))
).with_column(
    "CHURN_LABEL",
    F.when(
        F.col("COMPANY_RESPONSE_TO_CONSUMER").in_(
            F.lit("Closed with monetary relief"),
            F.lit("Closed without relief"),
            F.lit("Untimely response")
        ), F.lit(1)
    ).otherwise(F.lit(0))
).select(
    "ISSUE", "SUB_ISSUE", "SUB_PRODUCT", "STATE", "SUBMITTED_VIA",
    "TIMELY_RESPONSE", "CONSUMER_DISPUTED", "COMPANY",
    "DATE_RECEIVED", "DATE_SENT_TO_COMPANY",
    "CHURN_LABEL"
).fillna("UNKNOWN")

# 3. Behavioral Features
processed_df = processed_df.with_column(
    "IS_FEE_RELATED", F.iff(F.lower(F.col("ISSUE")).like("%fee%") | F.lower(F.col("ISSUE")).like("%interest%"), 1, 0)
).with_column(
    "IS_DISPUTE_RELATED", F.iff(F.lower(F.col("ISSUE")).like("%dispute%") | F.lower(F.col("ISSUE")).like("%purchase shown on your statement%"), 1, 0)
).with_column(
    "RESPONSE_DELAY_DAYS", F.datediff("day", F.col("DATE_RECEIVED"), F.col("DATE_SENT_TO_COMPANY"))
).with_column(
    "IS_TIMELY", F.iff(F.col("TIMELY_RESPONSE") == F.lit("Yes"), 1, 0)
).with_column(
    "IS_DISPUTED", F.iff(F.col("CONSUMER_DISPUTED") == F.lit("Yes"), 1, 0)
).drop(
    "DATE_RECEIVED", "DATE_SENT_TO_COMPANY", "TIMELY_RESPONSE", "CONSUMER_DISPUTED"
).with_column(
    "ROW_ID", F.seq8()
)

# 4. Materialize Gold Layer
processed_df.write.mode("overwrite").save_as_table(GOLD_TABLE)
session.sql(f"ALTER TABLE {GOLD_TABLE} SET CHANGE_TRACKING = TRUE").collect()

print(f"✅ Materialized {GOLD_TABLE}")

In [ ]:
fs = FeatureStore(session, DB, SCHEMA, default_warehouse="HACKATHON_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST)

# Define the Entity
entity = Entity(name="CUSTOMER", join_keys=["ROW_ID"])
fs.register_entity(entity)

# Register Feature View (dropping the label to keep features pure)
fv = FeatureView(
    name="CHURN_FEATURES_VIEW",
    entities=[entity],
    feature_df=session.table(GOLD_TABLE).drop("CHURN_LABEL"),
    desc="Dynamic churn features"
)
registered_fv = fs.register_feature_view(feature_view=fv, version="V1", overwrite=True)

# Generate versioned Dataset for training
ds_version = datetime.datetime.now().strftime("%Y%m%d_%H%M")
train_dataset = fs.generate_dataset(
    name=FS_DATASET_NAME,
    version=ds_version,
    spine_df=session.table(GOLD_TABLE).select("ROW_ID", "CHURN_LABEL"),
    features=[registered_fv],
    spine_label_cols=["CHURN_LABEL"]
)

train_df, test_df = train_dataset.read.to_snowpark_dataframe().random_split([0.8, 0.2])

In [ ]:
session.sql(f"USE DATABASE {DB}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"CREATE STAGE IF NOT EXISTS {DB}.{SCHEMA}.SNOWPARK_TEMP_STAGE").collect()

cat_cols = ["ISSUE", "SUB_ISSUE", "SUB_PRODUCT", "STATE", "SUBMITTED_VIA", "COMPANY"]
num_cols = ["IS_FEE_RELATED", "IS_DISPUTE_RELATED", "RESPONSE_DELAY_DAYS", "IS_TIMELY", "IS_DISPUTED"]
encoded_cols = [c+"_E" for c in cat_cols]
all_feature_cols = encoded_cols + num_cols

encoder = OrdinalEncoder(
    input_cols=cat_cols,
    output_cols=encoded_cols,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

model = XGBClassifier(
    input_cols=all_feature_cols,
    label_cols=["CHURN_LABEL"],
    output_cols=["PREDICTED_CHURN"],
    n_estimators=500,
    max_depth=12,
    learning_rate=0.05,
    scale_pos_weight=4.0,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1
)

pipeline = Pipeline(steps=[("encoder", encoder), ("model", model)])

trained_model = pipeline.fit(train_df)
preds = trained_model.predict(test_df)

test_pd = preds.to_pandas()

from sklearn.metrics import roc_auc_score as sk_roc_auc_score, recall_score as sk_recall_score, f1_score as sk_f1_score
import numpy as np

xgb_native = dict(trained_model.steps)["model"].to_xgboost()
feature_cols_ordered = all_feature_cols
X_test = test_pd[feature_cols_ordered].values
y_test = test_pd["CHURN_LABEL"].values

proba = xgb_native.predict_proba(X_test)[:, 1]
y_pred = test_pd["PREDICTED_CHURN"].values

auc_val = sk_roc_auc_score(y_test, proba)
recall_val = sk_recall_score(y_test, y_pred)
f1_val = sk_f1_score(y_test, y_pred)

metrics = {
    "auc": float(auc_val),
    "recall": float(recall_val),
    "f1": float(f1_val)
}

reg = Registry(session, database_name=DB, schema_name=SCHEMA)
mv = reg.log_model(
    model=trained_model,
    model_name=MODEL_NAME,
    version_name=f"V_{ds_version}",
    metrics=metrics,
    sample_input_data=test_df.limit(10)
)

importance_df = pd.DataFrame({
    "FEATURE": dict(trained_model.steps)["model"].get_input_cols(),
    "SCORE": xgb_native.feature_importances_
}).sort_values(by="SCORE", ascending=False)

session.create_dataframe(importance_df).write.mode("overwrite").save_as_table(f"{DB}.{SCHEMA}.MODEL_DRIVERS")

print(f"🚀 Model Logged! AUC: {metrics['auc']:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix as sk_confusion_matrix

cf_matrix = sk_confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cf_matrix, annot=True, fmt='g', cmap='Blues', cbar=False)
plt.title("Confusion Matrix: Customer Friction Prediction")
plt.xlabel("Predicted Risk")
plt.ylabel("Actual Risk")
plt.show()

importance_df_plot = importance_df.copy()
importance_df_plot['FEATURE'] = importance_df_plot['FEATURE'].str.replace("_E", "").str.replace("_", " ")

plt.figure(figsize=(10, 6))
sns.barplot(x="SCORE", y="FEATURE", data=importance_df_plot, hue="FEATURE", palette="magma", legend=False)
plt.title("Key Drivers of Customer Friction (Explainability)")
plt.xlabel("Importance Weight")
plt.ylabel(None)
plt.show()

print(f"""
Final Performance Report:
-------------------------
AUC Score: {metrics['auc']:.3f}
Recall:    {metrics['recall']:.3f}
F1 Score:  {metrics['f1']:.3f}
""")

In [ ]:
import shap.explainers._tree as _shap_tree
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

def _replacement_init(self, xgb_model):
    j = json.loads(xgb_model.save_raw(raw_format="json"))
    objective = j["learner"]["objective"]
    lmp = j["learner"]["learner_model_param"]
    gb = j["learner"]["gradient_booster"]
    if "gbtree" in gb:
        tree_info = gb["gbtree"]
    elif "dart" in gb:
        tree_info = gb["dart"]["gbtree"]
    else:
        tree_info = gb
    mp = tree_info.get("model_param", tree_info.get("tree_train_param", {}))
    n_targets = max(int(mp.get("num_output_group", 1)), 1)
    num_trees = len(tree_info.get("trees", []))
    if num_trees == 0:
        num_trees = int(mp.get("num_trees", 0))

    self.buf = xgb_model.save_raw()
    self.json = j
    self.num_trees = num_trees
    self.n_trees_per_iter = max(int(num_trees / n_targets), 1)
    self.n_targets = n_targets
    bs = lmp.get("base_score", "0.5")
    self.base_score = float(bs.strip("[]")) if isinstance(bs, str) and bs.startswith("[") else float(bs)
    self.name_obj = objective["name"]
    self.cat_feature_indices = []

_shap_tree.XGBTreeModelLoader.__init__ = _replacement_init

xgb_native = dict(trained_model.steps)["model"].to_xgboost()
feature_names = [c.replace("_E", "") for c in all_feature_cols]

X_sample_df = test_pd[all_feature_cols].sample(n=min(2000, len(test_pd)), random_state=42)
X_sample = X_sample_df.values.astype(np.float64)

explainer = shap.TreeExplainer(xgb_native)
shap_values = explainer.shap_values(X_sample)

shap_importance = pd.DataFrame({
    "FEATURE": feature_names,
    "MEAN_ABS_SHAP": np.abs(shap_values).mean(axis=0).round(6)
}).sort_values(by="MEAN_ABS_SHAP", ascending=False)

session.create_dataframe(shap_importance).write.mode("overwrite").save_as_table(f"{DB}.{SCHEMA}.SHAP_FEATURE_IMPORTANCE")

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.title("SHAP Feature Importance: Churn Drivers")
plt.tight_layout()
plt.show()

print("Top SHAP Drivers:")
print(shap_importance.to_string(index=False))

In [ ]:
test_pd["CHURN_PROBA"] = proba

segment_cols = ["STATE", "SUB_PRODUCT", "SUBMITTED_VIA"]
all_segments = []

for seg_col in segment_cols:
    seg = test_pd.groupby(seg_col).agg(
        TOTAL_COUNT=("CHURN_LABEL", "count"),
        CHURN_COUNT=("CHURN_LABEL", "sum"),
        AVG_CHURN_PROBA=("CHURN_PROBA", "mean")
    ).reset_index()
    seg["CHURN_RATE"] = (seg["CHURN_COUNT"] / seg["TOTAL_COUNT"]).round(4)
    seg["AVG_CHURN_PROBA"] = seg["AVG_CHURN_PROBA"].round(4)
    seg = seg.rename(columns={seg_col: "SEGMENT_VALUE"})
    seg["SEGMENT_TYPE"] = seg_col
    all_segments.append(seg[["SEGMENT_TYPE", "SEGMENT_VALUE", "TOTAL_COUNT", "CHURN_COUNT", "CHURN_RATE", "AVG_CHURN_PROBA"]])

segment_df = pd.concat(all_segments, ignore_index=True)
session.create_dataframe(segment_df).write.mode("overwrite").save_as_table(f"{DB}.{SCHEMA}.CHURN_SEGMENT_SCORES")

print(f"Saved {len(segment_df)} segment rows to {DB}.{SCHEMA}.CHURN_SEGMENT_SCORES")
print("\nTop 10 highest risk segments:")
print(segment_df.sort_values("AVG_CHURN_PROBA", ascending=False).head(10).to_string(index=False))